# Python 02 — Conditions and Loops

**Roadmap position:** Python → Core → Conditions and loops.

Work like an interview candidate: predict behaviour, implement manually, run focused checks, and explain failures. The examples use realistic validation and aggregation work rather than puzzle-only code.

## Outcome

You will be able to choose `if`/`elif`/`else` branches, iterate with `for` and `while`, use `break` and `continue` deliberately, and avoid common loop bugs.

## 1. Conditions express business rules

A condition must evaluate to `True` or `False`. Order matters: Python uses the first matching branch and skips the rest. Put specific rules before broader rules.

Use `==` to compare values. Use `is` mainly for singleton checks such as `value is None`, not for comparing strings or numbers.

In [1]:
def interview_readiness_status(completed_hours: float) -> str:
    """Return a readiness label for a non-negative study-hour total."""
    if completed_hours < 0:
        raise ValueError('completed_hours must not be negative')
    if completed_hours >= 100:
        return 'ready for mock interviews'
    if completed_hours >= 40:
        return 'building foundations'
    return 'getting started'

for hours in (0, 40, 99.5, 100):
    print(f'{hours:>5} hours -> {interview_readiness_status(hours)}')

    0 hours -> getting started
   40 hours -> building foundations
 99.5 hours -> building foundations
  100 hours -> ready for mock interviews


## Your turn 1 — classify an HTTP response

Implement `classify_http_status(status_code: int) -> str`. Return:

- `'success'` for codes 200–299;
- `'client error'` for codes 400–499;
- `'server error'` for codes 500–599;
- `'unexpected status'` for every other integer.

Do not list individual status codes. Use range comparisons. Consider the boundary values carefully.

In [8]:
# YOUR TURN
def classify_http_status(status_code: int) -> str:
    if status_code < 200 or status_code >599 or( status_code<400 and status_code>299):
        return ("unexpected status")
    if status_code >=200 and status_code <300:
        return ("success")
    if status_code >= 400 and status_code<500:
        return ("client error")
    if status_code >=500 and status_code<600:
        return ("server error")
                    
    


In [9]:
# Checks — run after implementing classify_http_status.
assert classify_http_status(200) == 'success'
assert classify_http_status(299) == 'success'
assert classify_http_status(400) == 'client error'
assert classify_http_status(499) == 'client error'
assert classify_http_status(500) == 'server error'
assert classify_http_status(599) == 'server error'
assert classify_http_status(302) == 'unexpected status'
print('HTTP-status checks passed.')

HTTP-status checks passed.


## 2. `for` loops: process each item

Use a `for` loop when you have an iterable—such as a list, string, dictionary, or `range`—and want to process each item. Keep the loop body focused. Accumulate results in clearly named variables.

`range(start, stop)` includes `start` but excludes `stop`; `range(1, 4)` produces `1, 2, 3`.

In [10]:
daily_study_hours = [1.5, 2.0, 0.0, 2.5, 3.0, 1.0, 0.5]
total_hours = 0.0

for hours in daily_study_hours:
    total_hours += hours

print(f'Weekly total: {total_hours} hours')
print(f'Days recorded: {len(daily_study_hours)}')

Weekly total: 10.5 hours
Days recorded: 7


## Your turn 2 — aggregate valid records

Implement `total_completed_hours`. Given a list of numbers, add only non-negative hours. Ignore negative values because they represent invalid records. Return the total as a `float`.

Use a `for` loop and an `if` statement. Do not use `sum`, comprehensions, or filtering helpers for this exercise.

In [11]:
# YOUR TURN
def total_completed_hours(hour_records: list[float]) -> float:
    total_hours=0.0
    for i in hour_records:
        if i > 0:
            total_hours+=i
            
    return total_hours        


In [12]:
# Checks — run after implementing total_completed_hours.
assert total_completed_hours([]) == 0.0
assert total_completed_hours([1.5, 2.0, 0.5]) == 4.0
assert total_completed_hours([1.5, -3.0, 2.0]) == 3.5
print('Aggregation checks passed.')

Aggregation checks passed.


## 3. `continue` and `break`

- `continue` skips the remainder of the current iteration and starts the next one. Use it when a record should be ignored.
- `break` ends the entire loop. Use it only when further work cannot improve the result, such as finding the first matching item.

Neither is inherently good or bad; the key question is whether future iterations are still needed.

In [13]:
application_ids = ['app-101', '', 'app-103', 'app-104']

for application_id in application_ids:
    if not application_id:
        print('Skipping empty ID')
        continue
    print(f'Processing {application_id}')
    if application_id == 'app-103':
        print('Target application found; stopping search.')
        break

Processing app-101
Skipping empty ID
Processing app-103
Target application found; stopping search.


## Debugging drill — the first-match bug

The function below should return the first job whose `status` is `'open'`, or `None` if there is no open job. It returns the wrong result.

1. Predict why the current `return` position fails.
2. Move the smallest amount of code necessary.
3. Test it with an empty list, a list whose first job is closed, and a list with no open jobs.

**Interview question:** Why is `return` inside a loop sometimes correct, and sometimes a bug?

In [17]:
# DEBUG ME
def find_first_open_job(jobs: list[dict[str, str]]) -> dict[str, str] | None:
    for job in jobs:
        if job['status'] == 'open':
            return job
        


jobs = [
    {'id': 'job-1', 'status': 'closed'},
    {'id': 'job-2', 'status': 'open'},
]
print(find_first_open_job(jobs))

{'id': 'job-2', 'status': 'open'}


## 4. `while` loops: repeat while a condition holds

Use `while` when the number of iterations is not known in advance. The condition must eventually become false, otherwise the loop never ends. In production code, retries should always have a limit and often a delay.

For this lesson, we simulate retries without network calls or delays.

In [18]:
max_attempts = 3
attempt_number = 1

while attempt_number <= max_attempts:
    print(f'Attempt {attempt_number} of {max_attempts}')
    attempt_number += 1

Attempt 1 of 3
Attempt 2 of 3
Attempt 3 of 3


## Your turn 3 — bounded retry simulation

Implement `attempt_connection`. It receives `outcomes`, a list of booleans where `True` means a connection succeeds, and `max_attempts`. Return the attempt number on the first success; return `None` when no success occurs within the allowed attempts.

Requirements:

- use a `while` loop;
- never index beyond the end of `outcomes`;
- never attempt more than `max_attempts`;
- stop immediately after the first success.

In [32]:
# YOUR TURN
def attempt_connection(outcomes: list[bool], max_attempts: int) -> int | None:
    attempts=1
    while attempts<max_attempts:
        if len(outcomes) == 0:
            return None
        if outcomes[attempts-1]== True:
            return attempts
        attempts+=1


In [33]:
# Checks — run after implementing attempt_connection.
assert attempt_connection([True], 3) == 1
assert attempt_connection([False, True], 3) == 2
assert attempt_connection([False, False, True], 2) is None
assert attempt_connection([], 3) is None
assert attempt_connection([False, False], 0) is None
print('Retry checks passed.')

Retry checks passed.


## Exit interview check

Before moving on, answer these without running code:

1. Why must specific conditions be checked before broader conditions?
2. When should you use `for` rather than `while`?
3. What is the difference between `continue`, `break`, and `return` inside a loop?
4. What creates an infinite `while` loop, and how do you prevent one?
5. Explain the time complexity of `total_completed_hours` and `find_first_open_job`.

When finished, send me your three implementations, the corrected debugging cell, and any failed test output. I’ll review them without editing your notebook. Then we will proceed to **functions and scope**.